# CLAP Embedding and Indexing
This notebook loads the dataset and runs **all** audio clips (both speech and non-speech) through the CLAP model to build a FAISS index.

It also includes the speech/non-speech labels from `audio_speech_labels.csv` in the index metadata.

In [ ]:
import os
import json
import glob
import torch
import faiss
import librosa
import numpy as np
import pandas as pd
import laion_clap
from tqdm.notebook import tqdm

In [ ]:
# --- Configuration Settings ---
import os
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = os.path.abspath('..')
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
INDEX_DIR = os.path.join(PROJECT_ROOT, "data_index")

os.makedirs(INDEX_DIR, exist_ok=True)

# CLAP Model Config
CLAP_SAMPLE_RATE = 48000
CHUNK_LENGTH_SECONDS = 10 
CHUNK_OVERLAP_SECONDS = 0 
CLAP_CHECKPOINT = "630k-audioset-fusion-best.pt"

# FAISS Index Config
FAISS_DIMENSION = 512
INDEX_FILENAME = "clap_embeddings.faiss"
METADATA_FILENAME = "clap_metadata.json"


In [ ]:
# --- Engine and Indexer Classes ---
class CLAPEngine:
    def __init__(self, use_cuda=True):
        self.device = torch.device('cuda' if use_cuda and torch.cuda.is_available() else 'cpu')
        print(f"Initializing CLAP Recommender on device: {self.device}")
        
        self.model = laion_clap.CLAP_Module(enable_fusion=False, device=self.device)
        
        checkpoint_path = os.path.join(PROJECT_ROOT, "models", CLAP_CHECKPOINT)
        if os.path.exists(checkpoint_path):
            print(f"Loading CLAP checkpoint from {checkpoint_path}")
            self.model.load_ckpt(checkpoint_path)
        else:
            print("Downloading default CLAP checkpoint...")
            self.model.load_ckpt()
            
        self.model.eval()
        
    def _chunk_audio(self, audio_data: np.ndarray) -> list:
        chunk_length_samples = CHUNK_LENGTH_SECONDS * CLAP_SAMPLE_RATE
        overlap_samples = CHUNK_OVERLAP_SECONDS * CLAP_SAMPLE_RATE
        step = chunk_length_samples - overlap_samples
        
        if step <= 0:
            raise ValueError("CHUNK_OVERLAP_SECONDS must be strictly less than CHUNK_LENGTH_SECONDS")
            
        total_samples = len(audio_data)
        chunks = []
        
        if total_samples <= chunk_length_samples:
            return [audio_data]
            
        for start in range(0, total_samples, step):
            end = start + chunk_length_samples
            chunks.append(audio_data[start:end])
            if end >= total_samples:
                break
                
        return chunks

    def get_clip_embedding(self, filepath: str) -> np.ndarray:
        try:
            audio_data, _ = librosa.load(filepath, sr=CLAP_SAMPLE_RATE)
            chunks = self._chunk_audio(audio_data)
            embeddings = self.model.get_audio_embedding_from_data(x=chunks, use_tensor=False)
            mean_embedding = np.mean(embeddings, axis=0)
            
            norm = np.linalg.norm(mean_embedding)
            if norm > 0:
                mean_embedding = mean_embedding / norm
                
            return mean_embedding
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            return None

class AudioIndexer:
    def __init__(self):
        self.index_path = os.path.join(INDEX_DIR, INDEX_FILENAME)
        self.metadata_path = os.path.join(INDEX_DIR, METADATA_FILENAME)
        
        self.index = faiss.IndexFlatIP(FAISS_DIMENSION)
        self.metadata = []
        
    def add_embedding(self, filename: str, embedding: np.ndarray, is_speech: bool):
        if len(embedding.shape) == 1:
            embedding = embedding.reshape(1, -1)
            
        embedding = embedding.astype('float32')
        self.index.add(embedding)
        
        record = {
            "id": len(self.metadata),
            "filename": filename,
            "is_speech": is_speech
        }
        self.metadata.append(record)
        
    def save(self):
        faiss.write_index(self.index, self.index_path)
        with open(self.metadata_path, 'w') as f:
            json.dump(self.metadata, f, indent=4)
        print(f"Saved FAISS index with {self.index.ntotal} records to {self.index_path}")

In [ ]:
# --- 1. Load CSV and build metadata map ---
csv_path = os.path.join(DATA_DIR, 'audio_speech_labels.csv')
df = pd.read_csv(csv_path)

ID_COLUMN = os.getenv("ID_COLUMN", "id")

# Create a dictionary mapping the string ID to its 'is_speech' boolean
file_metadata = dict(zip(df[ID_COLUMN].astype(str), df['is_speech']))
all_valid_ids = set(file_metadata.keys())

print(f"Found {len(all_valid_ids)} total records in CSV.")


In [ ]:
# --- 2. Initialize Engine and Indexer ---
engine = CLAPEngine()
indexer = AudioIndexer()

In [ ]:
# --- 3. Discover downloaded audio files ---
audio_dir = os.path.join(DATA_DIR, "audio")
audio_files = glob.glob(os.path.join(audio_dir, "**", "*.*"), recursive=True)

# Match the file name without extension against the set of IDs
valid_audio_files = [f for f in audio_files if os.path.splitext(os.path.basename(f))[0] in all_valid_ids]

print(f"Found {len(valid_audio_files)} downloaded audio files ready for processing.")


In [ ]:
# --- 4. Process and Index ---
failed_files = []
processed = 0

for filepath in tqdm(valid_audio_files, desc="Extracting CLAP Embeddings"):
    filename = os.path.basename(filepath)
    file_id = os.path.splitext(filename)[0]
    
    # Get speech label, default to False if missing for some reason
    is_speech = bool(file_metadata.get(file_id, False))
    
    embedding = engine.get_clip_embedding(filepath)
    if embedding is not None:
        indexer.add_embedding(filename, embedding, is_speech)
        processed += 1
    else:
        failed_files.append(filename)

print(f"Successfully indexed {processed} files!")
if failed_files:
    print(f"Failed to process {len(failed_files)} files.")


In [ ]:
# --- 5. Save the final FAISS index and metadata ---
indexer.save()